# 04 — Heaps and Beam Search
## Keeping the best candidates without sorting everything


**Rule:** every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell.
- **`assert` lines** — the specification. If one fails after you edit a cell, your change broke a rule.

In [1]:
from __future__ import annotations

import math, random, time
from dataclasses import dataclass, field
from typing import Callable, Iterable

SEED = 7
random.seed(SEED)
print(f"Ready. Seed = {SEED}")

Ready. Seed = 7


## Before we start: two ideas

**A heap** is a data structure that keeps the smallest item instantly reachable. Python's `heapq` is a *min-heap*: the smallest value is always at `heap[0]`, and adding or removing costs `O(log n)`.

Trick: to keep the **largest** `k` values from a long stream, hold a min-heap of size `k`. Its smallest element (`heap[0]`) is the weakest of your current top `k` — the first to kick out when something better arrives.

**Beam search** uses the same "keep only the best few" idea to generate sequences. At each step: every unfinished sequence proposes next tokens, you score all the resulting candidates, keep only the best `width`, and repeat. It is **not** a full search — a sequence dropped early can never come back — so `width` trades quality for cost.

This notebook connects both to log-probabilities, a length penalty, deterministic tie-breaking, and a measured cost comparison.

## 1. Keeping the top `k` of a stream

**Predict.**
- `top_k_stream([5, 1, 9, 3, 7, 8], 3)` → ?
- `top_k_stream([...], 0)` → ?

![Bounded heap](assets/heap.svg)

In [2]:
import heapq

def top_k_stream(values: Iterable[float], k: int) -> list[float]:
    if k < 0:
        raise ValueError("k must be non-negative")
    heap: list[float] = []
    for value in values:
        if len(heap) < k:
            heapq.heappush(heap, value)                 # still filling up
        elif k and value > heap[0]:
            heapq.heapreplace(heap, value)             # better than the weakest -> swap
    return sorted(heap, reverse=True)

print("top 3 of [5, 1, 9, 3, 7, 8]:", top_k_stream([5, 1, 9, 3, 7, 8], 3))
print("top 0 (k = 0)              :", top_k_stream([3, 1, 2], 0))

assert top_k_stream([5, 1, 9, 3, 7, 8], 3) == [9, 8, 7]
assert top_k_stream([3, 1, 2], 0) == []

top 3 of [5, 1, 9, 3, 7, 8]: [9, 8, 7]
top 0 (k = 0)              : []


In [3]:
try:
    top_k_stream([1, 2], -1)
except ValueError as exc:
    print("negative k refused:", exc)
else:
    raise AssertionError("Expected negative k to be rejected")

negative k refused: k must be non-negative


### What you just saw

The heap never holds more than `k` items, so memory is `O(k)` even for an endless stream. `heap[0]` is the weakest of the current top `k`; when a bigger value arrives you replace just that one. The final `sorted(...)` is only to present the result in order — it is not needed while consuming the stream.

`k = 0` returns an empty list; a negative `k` is meaningless and raises.

## 2. Why language-model scores use logarithms

A sequence's probability is the product of its token probabilities. Multiply enough numbers below 1 and you underflow to 0.

Take logarithms: a product becomes a **sum**, and the numbers stay in a sane range. Every `log(p)` with `p <= 1` is `<= 0`, so longer sequences pile up more negative score even when every token was a good choice. Decoders fix that bias with a **length penalty** — a tunable knob, not a fact of nature.

In [4]:
@dataclass(order=True)
class Beam:
    score: float                                        # only this field is compared
    tokens: tuple[str, ...] = field(compare=False)
    log_probability: float = field(compare=False)
    finished: bool = field(default=False, compare=False)

def normalized_score(logp: float, length: int, alpha: float = 0.7) -> float:
    """Raw log-prob divided by length**alpha. alpha=0 -> no penalty; alpha=1 -> divide by length."""
    return logp / max(length, 1) ** alpha

def beam_search(next_tokens: Callable[[tuple[str, ...]], list[tuple[str, float]]],
                width: int = 3, max_steps: int = 5, eos: str = "<eos>") -> list[Beam]:
    if width <= 0:
        raise ValueError("width must be positive")
    if max_steps < 0:
        raise ValueError("max_steps must be non-negative")

    beams = [Beam(0.0, (), 0.0, False)]
    for _ in range(max_steps):
        candidates = []
        for beam in beams:
            if beam.finished:
                candidates.append(beam)                  # keep finished beams, do not extend them
                continue
            for token, probability in next_tokens(beam.tokens):
                if not 0.0 < probability <= 1.0:
                    raise ValueError("token probabilities must be in (0, 1]")
                logp = beam.log_probability + math.log(probability)
                tokens = beam.tokens + (token,)
                candidates.append(Beam(normalized_score(logp, len(tokens)), tokens, logp, token == eos))
        beams = heapq.nlargest(width, candidates)        # keep the best `width`
        if all(b.finished for b in beams):
            break
    return sorted(beams, reverse=True)

print("beam_search defined")

beam_search defined


### The scoring, in words

For token probabilities $p_1, \ldots, p_L$:

$$
\log P = \sum_{i=1}^{L} \log(p_i)
$$

`normalized_score` divides that by `length ** alpha` so you can dial the length bias. It does not make beam search exact — pruning still happens locally at each step.

Finished beams (ending in `<eos>`) are carried along unchanged; unfinished ones get extended.

**Predict.** With the `tiny_model` below and `width=2`, which two sequences come back, and which scores higher?

In [5]:
def tiny_model(prefix: tuple[str, ...]) -> list[tuple[str, float]]:
    if not prefix:            return [("AI", .55), ("Data", .45)]
    if prefix[-1] == "AI":    return [("helps", .7), ("<eos>", .3)]
    if prefix[-1] == "Data":  return [("matters", .8), ("<eos>", .2)]
    return [("<eos>", 1.0)]

for beam in beam_search(tiny_model, width=2):
    print(f"{beam.tokens}  score={beam.score:.3f}  raw log-prob={beam.log_probability:.3f}  finished={beam.finished}")

('AI', 'helps', '<eos>')  score=-0.442  raw log-prob=-0.955  finished=True
('Data', 'matters', '<eos>')  score=-0.473  raw log-prob=-1.022  finished=True


### Reading that output

You get at most `width` beams back. Each carries its length-normalised `score`, its `tokens`, the raw cumulative `log_probability`, and whether it is `finished`.

`width=1` would behave like greedy decoding (take the single best token each step). Bigger `width` keeps more options but calls `next_tokens` more and uses more memory. Because pruning is per-step, beam search can still miss the globally best sequence.

In [6]:
# Both returned beams are finished.
print("all finished:", all(b.finished for b in beam_search(tiny_model, width=2)))

# Bad parameters are refused.
for w, steps in ((0, 5), (-1, 5), (1, -1)):
    try:
        beam_search(tiny_model, width=w, max_steps=steps)
    except ValueError as exc:
        print(f"width={w}, max_steps={steps} -> {exc}")
    else:
        raise AssertionError("bad parameters should raise")

# A probability of 0 is illegal (log(0) is -inf).
def broken_model(prefix): return [("bad", 0.0)]
try:
    beam_search(broken_model, width=1)
except ValueError as exc:
    print("probability 0.0 ->", exc)
else:
    raise AssertionError("probability 0 should raise")

all finished: True
width=0, max_steps=5 -> width must be positive
width=-1, max_steps=5 -> width must be positive
width=1, max_steps=-1 -> max_steps must be non-negative
probability 0.0 -> token probabilities must be in (0, 1]


## 3. Breaking ties on purpose

When you push tuples onto a heap and two priorities are equal, Python compares the **next** element of the tuple. If that is a dict or a custom object, the comparison can raise `TypeError`.

Fix: put a always-increasing counter as the second element. It is unique, it is comparable, and it makes equal-priority order deterministic (first in, first out).

In [7]:
from itertools import count

counter = count()
queue = []
for priority, payload in [(1, {"job": "A"}), (1, {"job": "B"})]:
    heapq.heappush(queue, (priority, next(counter), payload))    # (priority, tiebreak, data)

first_out = heapq.heappop(queue)[2]["job"]
print("both priority 1; first popped:", first_out, " <- 'A', because it was pushed first")
assert first_out == "A"

both priority 1; first popped: A  <- 'A', because it was pushed first


### Note

The counter is **not** part of the score. It only decides ties. `beam_search` above does not need it because `heapq.nlargest` already breaks ties by insertion order — you need the counter when you push onto a heap yourself.

## 4. Is the heap actually faster?

Full sort is `O(n log n)`. `heapq.nlargest(k, ...)` is about `O(n log k)`, which is better when `k` is small. Let's measure it — and then check whether it matters.

In [8]:
rng = random.Random(SEED)
values = [rng.random() for _ in range(100_000)]

# First: correctness. The streaming top-k must equal a full sort's top-k.
assert top_k_stream(values, 10) == sorted(values, reverse=True)[:10]
print("streaming top-10 matches full-sort top-10")

streaming top-10 matches full-sort top-10


In [9]:
def best_of(fn, *args, repeats=5):
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - start)
    return best

stream_ms = best_of(lambda: top_k_stream(values, 10)) * 1e3
sort_ms = best_of(lambda: sorted(values, reverse=True)[:10]) * 1e3
print(f"top_k_stream: {stream_ms:.1f} ms   |   full sort: {sort_ms:.1f} ms   (n={len(values):,}, k=10)")
print("The heap is usually faster here -- but both are tiny next to real model inference,")
print("so a cheaper selection step does not always shorten total latency.")

top_k_stream: 10.4 ms   |   full sort: 31.7 ms   (n=100,000, k=10)
The heap is usually faster here -- but both are tiny next to real model inference,
so a cheaper selection step does not always shorten total latency.


### What you just saw

The `assert` checks *correctness* (same top-10 as a real sort), not speed. The timing cell reports the best of a few runs. A serious study would sweep both `n` and `k` and report a distribution, not one number — and would measure end to end, because model inference usually dwarfs this bookkeeping.

## Project — Beam decoder with constraints

Implement a beam decoder that combines log probabilities, a configurable length penalty, EOS handling, a trie-based token constraint and deterministic tie-breaking.

**Suggested test matrix:**

- `width=1` agrees with greedy decoding for the same scoring rule.
- Beam width limits the number of live candidates at every step.
- Finished beams are retained and are never expanded after EOS.
- Token probabilities outside `(0, 1]` and invalid widths fail clearly.
- Illegal tokens never appear in a decoded sequence.
- A dead-end constraint produces a diagnostic or an explicit EOS policy.
- Equal scores produce deterministic output across repeated runs.
- Length penalty changes ranking in a documented, testable way.
- The decoder preserves cumulative log probability separately from normalized score.
- Runtime and memory are reported as beam width grows.

**Acceptance criteria:** width 1 matches greedy decoding; illegal tokens never appear; completed beams are retained; tests cover ties and dead ends; report time and memory as width grows.

You may `from course_utils import Beam, normalized_score, beam_search, TokenTrie` instead of copying code from this notebook and notebook 03.

**Checks to run yourself**

- Run `width=1` and a hand-written greedy loop on the same model; assert the token sequences match.
- Count `next_tokens` calls per step and assert it never exceeds `width × branching`.
- Feed a model whose arg-max token is banned by the trie; assert it never appears in the output.
- Run the decoder twice on a model with tied probabilities; assert identical output.
- Sweep `width` in `(1, 2, 4, 8)` and print decode time and peak candidate count.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse Beam + normalized_score from this notebook and TokenTrie from notebook 03,
# or import them:
#     from course_utils import Beam, normalized_score, TokenTrie
#
# 1. constrained_beam_search(next_tokens, trie, width, max_steps, eos, alpha):
#      - expand only tokens in trie.allowed_next(beam.tokens)
#      - keep cumulative log-prob separate from the length-normalised score
#      - deterministic tie-breaking (a monotonic counter, or heapq.nlargest's own order)
#      - carry finished beams forward unchanged
# 2. Dead-end policy when allowed_next is empty: emit EOS if legal, else raise a diagnostic.
# 3. Tests: the matrix above.

def constrained_beam_search(next_tokens, trie, width=3, max_steps=8, eos="<eos>", alpha=0.7):
    raise NotImplementedError("Implement the constrained beam decoder")
